# Using OpenAI models on Amazon Bedrock with Strands Agents

## Overview

OpenAI's GPT-5 models are available on Amazon Bedrock, callable through the [OpenAI Responses API](https://strandsagents.com/docs/user-guide/concepts/model-providers/openai-responses/). Strands' `OpenAIResponsesModel` provider talks to that API, so you can run an agent against an OpenAI model using AWS-native authentication and without a separate OpenAI account.

In this example we use `openai.gpt-5.4` on Amazon Bedrock as the agent's model, with a simple `current_time` and `current_weather` tool use case. We cover both ways to authenticate — a Bedrock API key and a short-term generated token — below.

## Tutorial Details

| Information      | Details                                                            |
|:-----------------|:------------------------------------------------------------------|
| Agent structure  | Single agent                                                      |
| Model provider   | OpenAI Responses (`OpenAIResponsesModel`)                          |
| Model            | `openai.gpt-5.4` on Amazon Bedrock                                 |
| API              | OpenAI Responses API, reached through the Amazon Bedrock endpoint  |
| Custom tools     | current_time, current_weather                                     |
| Strands features | Calling OpenAI-on-Bedrock models with the Responses API provider   |

## Architecture

<div style="text-align:center">
    <img src="images/simple_agent.png" width="65%" />
</div>

## What you'll learn
* Configure an OpenAI-on-Bedrock model with the `OpenAIResponsesModel` provider
* Authenticate with a Bedrock API key or a short-term generated token
* Give the agent custom tools and inspect its response metrics

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Access to OpenAI models (e.g. `openai.gpt-5.4`) on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)

Install the required packages for our agent:

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages:

In [4]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.openai_responses import OpenAIResponsesModel


### Set up credentials

To call OpenAI models on Amazon Bedrock you need AWS credentials. You have two options:

- **Bedrock API key**: Generate a short-term bearer token from the AWS Management Console and set it as `AWS_BEARER_TOKEN_BEDROCK`. This is the quickest way to get started.
- **AWS credentials**: Set your standard AWS credentials and let the [aws-bedrock-token-generator](https://pypi.org/project/aws-bedrock-token-generator/) package mint a short-term token for you at request time.

Fill in whichever option you plan to use. Both approaches are demonstrated below.

In [ ]:
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "<YOUR_BEDROCK_API_KEY>"

os.environ["AWS_REGION"] = "<AWS_REGION>"
os.environ["AWS_ACCESS_KEY_ID"] = "<YOUR_AWS_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<YOUR_AWS_SECRET_ACCESS_KEY>"
os.environ["AWS_SESSION_TOKEN"] = "<YOUR_AWS_SESSION_TOKEN>"

region = os.environ["AWS_REGION"]

# uncomment next line to test:
#!aws sts get-caller-identity

### Setting up custom tools

Let's now set up two tools to test our agent:

In [6]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Replace with actual weather API call.
    return "sunny"

### Create the agent with a Bedrock API key

Configure the `OpenAIResponsesModel` to reach the Amazon Bedrock endpoint. Two `client_args` do the work: `base_url` points at the Bedrock endpoint for OpenAI models (`https://bedrock-mantle.{region}.api.aws/openai/v1`), and `api_key` carries your Bedrock API key. Run the next cell if you set `AWS_BEARER_TOKEN_BEDROCK`; otherwise skip to the token-generator approach below.

In [8]:
api_key = os.environ["AWS_BEARER_TOKEN_BEDROCK"]

model = OpenAIResponsesModel(
    model_id="openai.gpt-5.4",
    client_args={
        "api_key": api_key,
        "base_url": f"https://bedrock-mantle.{region}.api.aws/openai/v1",
    },
)

agent = Agent(model=model)
response = agent("What is the capital of British Columbia?")

Victoria.

### Alternative: mint a short-term token from AWS credentials

If you are using standard AWS credentials instead of a Bedrock API key, use `provide_token` from `aws-bedrock-token-generator` to mint a short-term bearer token from your credentials. Here we also give the agent the `current_time` and `current_weather` tools.

In [9]:
from aws_bedrock_token_generator import provide_token

model = OpenAIResponsesModel(
    model_id="openai.gpt-5.4",
    client_args={
        "api_key": provide_token(region=region),
        "base_url": f"https://bedrock-mantle.{region}.api.aws/openai/v1",
    },
)

system_prompt = "You are a simple agent that can tell the time and the weather"

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[current_time, current_weather],
)

response = agent("What is the time and weather in Philadelphia?")



Tool #1: current_time

Tool #2: current_weather
In Philadelphia, it’s currently 5:13 PM local time, and the weather is sunny.

### Shorthand: let the provider handle the Bedrock endpoint

Building the `base_url` and passing a token by hand works, but `OpenAIResponsesModel` can do both for you. Pass `bedrock_mantle_config` and the provider derives the correct Bedrock endpoint from your region and mints a fresh token for each request from your AWS credentials:

```python
from strands.models.openai_responses import OpenAIResponsesModel

model = OpenAIResponsesModel(
    bedrock_mantle_config={"region": region},  # region is optional; falls back to the AWS credential chain
    model_id="openai.gpt-5.4",
)
```

When you set `bedrock_mantle_config`, do not also pass `base_url` or `api_key` in `client_args` — the provider derives them and raises a `ValueError` if they are present.

Let's take a look at the usage of our agent for the last query by examining the response `metrics`:

In [ ]:
from pprint import pprint

pprint(vars(response.metrics))

{'accumulated_metrics': {'latencyMs': 0},
 'accumulated_usage': {'inputTokens': 284,
                       'outputTokens': 76,
                       'totalTokens': 360},
 'agent_invocations': [AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='f265095a-7206-4416-b812-fa6ca3f2710d',
                                                                    usage={'inputTokens': 96,
                                                                           'outputTokens': 52,
                                                                           'totalTokens': 148}),
                                               EventLoopCycleMetric(event_loop_cycle_id='04223e1d-4a02-43bb-a77c-8e90be7bd24e',
                                                                    usage={'inputTokens': 188,
                                                                           'outputTokens': 24,
                                                                           'totalTokens': 212})]

## Summary

In this notebook you learned how to call an OpenAI model hosted on Amazon Bedrock with the `OpenAIResponsesModel` provider and the Responses API. You authenticated with both a Bedrock API key and a short-term generated token, gave the agent custom tools, and inspected its response metrics. This completes the model providers tutorial: you have now run a Strands agent against a local Ollama model, an Azure OpenAI model through LiteLLM, and an OpenAI model on Amazon Bedrock.